# Abstract

This study uses an explicitly synthetic structural causal model to examine a training-participation claim. The intervention sets participation to one versus zero, and the estimand is the total population average effect on a synthetic score. The causal graph includes observed baseline variables, hidden ability, a mediator, and a post-treatment collider. The known population effect is 3.06 points; the 2500-record sample has standardized truth 3.066.

Adjustment for observed baseline variables estimates 4.179, while an oracle regression that also observes hidden ability estimates 3.008. The first result is an adjusted association, because hidden confounding prevents identification from the observed variables. Mediator and collider adjustments illustrate different failures of an indiscriminate control strategy. A pre-treatment negative-control outcome has an observed adjusted participation contrast of 0.790, despite having no causal participation effect.

Two hundred independent replications at each of four confounding strengths retain bias, robust-interval coverage, and Monte Carlo uncertainty. At confounding strength one, observed-adjustment coverage is 0.0%; the oracle provides a simulation benchmark rather than a deployable remedy. Group-specific effects and hypothetical costs demonstrate that an aggregate benefit does not settle allocation or fairness. The decision memo rejects automated allocation from unidentified associations and specifies equitable evaluation, human appeal, and stop conditions. These results concern the stated data-generating model only: there are no real participants, measured economic benefits, or empirical claims about actual training programs.


# DS-A6: Causal Claim and Sensitivity

**Author:** xuhongbo · **Student ID:** 202618018629048

Use an explicitly synthetic training-participation example with known potential outcomes. The data are generated locally from the published structural equations and frozen seed; no row represents a real person. The main lesson is the downgrade from a causal claim to an adjusted association when hidden confounding prevents identification. The oracle is a simulation validation device, not a proposed observed-data estimator.

## 01 · Frozen protocol and provenance

Verify the raw-file hashes before analysis. Every run creates a new directory and records its environment. No automatic execution is represented as student independent verification.

Run from the repository directory after installing `requirements.txt`. Run this notebook in order, or run `python analysis.py`; each execution retains a new results directory. Use `python verify_results.py` after saving a completed notebook. The fixed protocol is retained in `results/20261010T104323297530Z/protocol.json`. `generate_data.py` supplies the synthetic generator; analysis verifies the archived sample without overwriting the raw data.

In [1]:
from pathlib import Path
import os, sys, json, hashlib, datetime, subprocess, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1)
plt.rcParams.update({'figure.dpi':110, 'font.size':10, 'axes.spines.top':False, 'axes.spines.right':False})
BASE = Path.cwd()
PROTOCOL_PATH = BASE / 'results/20261010T104323297530Z/protocol.json'
assert PROTOCOL_PATH.exists(), 'Use the DS-A6 directory as the working directory.'
protocol = json.loads(PROTOCOL_PATH.read_text())
def notebook_code_hash():
    saved=json.loads((BASE/f"A{protocol['assignment']}.ipynb").read_text())
    code=''.join(''.join(c['source']) for c in saved['cells'] if c['cell_type']=='code')
    return hashlib.sha256(code.encode()).hexdigest()
SEED = protocol['seed']
rng = np.random.default_rng(SEED)
RUN = BASE / 'results' / datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN.mkdir(parents=True)
(RUN/'figures').mkdir()
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def save_json(name, value):
    (RUN/name).write_text(json.dumps(value, indent=2, default=lambda x: x.item() if hasattr(x,'item') else str(x))+'\n')
def table(name, value):
    value.to_csv(RUN/(name+'.csv'), index=False)
    display(value.head(20))
def figure(name):
    plt.tight_layout(); plt.savefig(RUN/'figures'/(name+'.png'), dpi=150, bbox_inches='tight'); plt.show(); plt.close()
manifest = json.loads((BASE/'data/raw/manifest.json').read_text())
for filename, expected in manifest['files'].items():
    assert sha(BASE/'data/raw'/filename)==expected, filename
save_json('run_metadata.json', {'started_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(), 'python':sys.version,'executable':sys.executable,'platform':platform.platform(),'protocol_sha256':sha(PROTOCOL_PATH),'manifest_sha256':sha(BASE/'data/raw/manifest.json'),'seed':SEED,'analysis_source_sha256':sha(BASE/'analysis.py'),'notebook_code_sha256':notebook_code_hash()})
(RUN/'protocol.json').write_bytes(PROTOCOL_PATH.read_bytes())
(RUN/'requirements-lock.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print('Verified raw hashes:', len(manifest['files']))
print('Run:', RUN.name)


Verified raw hashes: 3
Run: 20261125T111718437488Z


## 02 · Intervention, structural model and DAG

The intervention sets training participation A to 1 versus 0. Y is a synthetic task score, and the target is the total population ATE E[Y(1)−Y(0)]. Baseline X and binary group G are observed; U is hidden ability. The mediator M follows participation, and C is a post-treatment collider A → C ← U. Equations are A ~ Bernoulli(expit(−0.2 + 0.6X + 0.4G + γU)); M = 0.7A + 0.4X + 0.6U + εM; Y = (2+G)A + 0.8M + 1.5X + G + U + εY. With G ~ Bernoulli(0.5), the total individual effect is 2.56+G and population ATE is 3.06. Negative-control Ypre = 0.8X + 0.5G + U + εpre precedes A and cannot be affected by it.

In [2]:

from generate_data import generate
from matplotlib.patches import FancyArrowPatch
observed=pd.read_csv(BASE/'data/raw/observed.csv'); oracle=pd.read_csv(BASE/'data/raw/oracle.csv')
assert np.allclose(observed,oracle[observed.columns])
assert np.allclose(oracle.Y1-oracle.Y0,2.56+oracle.G)
regenerated=generate(protocol['n'],protocol['confounding_strength'],SEED)
assert np.allclose(regenerated,oracle,rtol=1e-12,atol=1e-12)
truth=float((oracle.Y1-oracle.Y0).mean()); population_truth=3.06
print('Synthetic rows:',len(observed),'Known sample-standardized ATE:',truth,'Population ATE:',population_truth)
positions={'X':(0,3),'G':(0,1),'U':(2,4),'Ypre':(4,4),'A':(2,2),'M':(4,2),'C':(2,0),'Y':(6,2)}
edges=[('X','A'),('X','M'),('X','Y'),('X','Ypre'),('G','A'),('G','Y'),('G','Ypre'),('U','A'),('U','M'),('U','C'),('U','Y'),('U','Ypre'),('A','M'),('A','C'),('A','Y'),('M','Y')]
fig,ax=plt.subplots(figsize=(11,6)); ax.set(xlim=(-.7,6.7),ylim=(-.7,4.7)); ax.axis('off')
labels={'X':'X\nbaseline','G':'G\ngroup','U':'U\nhidden','A':'A\nparticipation','M':'M\nmediator','C':'C\ncollider','Y':'Y\nscore','Ypre':'Ypre\npre-outcome'}
boxes={}
for node,pos in positions.items():
    boxes[node]=ax.text(*pos,labels[node],ha='center',va='center',fontsize=10,zorder=3,bbox={'boxstyle':'round,pad=.55','facecolor':{'U':'#eeeeee','M':'#fff0bd','C':'#ffd6d6','A':'#cce5ff'}.get(node,'white'),'edgecolor':'black','linestyle':'--' if node=='U' else '-'})
fig.canvas.draw()
for src,dst in edges:
    rad={('U','C'):.6,('X','Y'):-.3,('G','Y'):.25,('U','Y'):-.15,('A','Y'):.22}.get((src,dst),-.06)
    arrow=FancyArrowPatch(positions[src],positions[dst],patchA=boxes[src].get_bbox_patch(),patchB=boxes[dst].get_bbox_patch(),arrowstyle='-|>',mutation_scale=15,shrinkA=3,shrinkB=3,connectionstyle=f'arc3,rad={rad}',color='#666666',linestyle='--' if src=='U' else '-',lw=1.3,zorder=2)
    ax.add_patch(arrow)
ax.set_title('Synthetic causal DAG: U is hidden; M and C are post-treatment'); figure('causal_dag')
pd.DataFrame(edges,columns=['source','target']).to_csv(RUN/'dag_edges.csv',index=False)


Synthetic rows: 2500 Known sample-standardized ATE: 3.0664 Population ATE: 3.06


## 03 · Identification and adjustment sets

Under consistency and no interference, adjustment for X,G,U identifies the total effect in this structural model. X,G alone do not block A ← U → Y or A ← U → M → Y when γ > 0. The observed-data contrast is therefore labelled an adjusted association, not an identified causal estimate. Ypre is only a noisy proxy and negative-control outcome; it is not assumed sufficient to replace U. M blocks part of the total effect, while conditioning on C can open collider bias. Oracle adjustment uses hidden U only to validate the simulation and is not a feasible real-data solution.

In [3]:

assumptions=pd.DataFrame([
 {'assumption':'Consistency and no interference','status':'True by construction in this simulation','implication':'A fixes one specified intervention; records do not affect each other'},
 {'assumption':'Exchangeability given X,G','status':'False for gamma > 0','implication':'Observed adjusted association cannot identify total ATE'},
 {'assumption':'Exchangeability given X,G,U','status':'True by structural design','implication':'Oracle total-effect check, not an available estimator'},
 {'assumption':'Positivity','status':'Logistic propensity strictly between 0 and 1','implication':'Finite-sample near-tail overlap remains a precision concern'},
 {'assumption':'No post-treatment adjustment for total effect','status':'Violated by including M or C','implication':'Mediator changes estimand; collider may introduce bias'},
 {'assumption':'Measurement and timing','status':'Known synthetic generation order','implication':'No evidence about actual training programs or participants'}])
table('assumptions',assumptions)
p=oracle.true_propensity
save_json('overlap.json',{'true_propensity_min':float(p.min()),'true_propensity_max':float(p.max()),'fraction_outside_005_095':float(((p<.05)|(p>.95)).mean()),'oracle_only':True})
plt.figure(figsize=(7,3.5))
for a in [0,1]: plt.hist(p[oracle.A==a],bins=np.linspace(0,1,21),alpha=.5,label=f'A={a}')
plt.xlabel('True propensity (oracle diagnostic)'); plt.ylabel('Records'); plt.legend(); figure('overlap')


,assumption,status,implication
0,Consistency and no interference,True by construction in this simulation,A fixes one specified intervention; records do...
1,"Exchangeability given X,G",False for gamma > 0,Observed adjusted association cannot identify ...
2,"Exchangeability given X,G,U",True by structural design,"Oracle total-effect check, not an available es..."
3,Positivity,Logistic propensity strictly between 0 and 1,Finite-sample near-tail overlap remains a prec...
4,No post-treatment adjustment for total effect,Violated by including M or C,Mediator changes estimand; collider may introd...
5,Measurement and timing,Known synthetic generation order,No evidence about actual training programs or ...


## 04 · Estimation, robust intervals and bad-control comparisons

Fit linear outcome regressions with an A×G interaction and average their A contrasts over the sample G distribution. HC3 covariance is (ZᵀZ)⁻¹ Zᵀ diag[eᵢ²/(1−hᵢ)²] Z (ZᵀZ)⁻¹; the standardized contrast uses gradient (0,1,0,0,mean(G),…). Intervals use a normal reference and condition on the observed covariate distribution. Robust standard errors address variance misspecification, not confounding bias. The sample-standardized causal truth differs slightly from population 3.06 because the realized group fraction need not equal one half. For the oracle mediator-controlled fit, the appropriate direct-effect target is 2+mean(G), not the total effect.

In [4]:

def estimate(data,extra=(),outcome='Y',group_mean=None):
    Z=np.column_stack([np.ones(len(data)),data.A,data.X,data.G,data.A*data.G]+[data[c] for c in extra])
    yy=data[outcome].to_numpy(); beta=np.linalg.lstsq(Z,yy,rcond=None)[0]
    bread=np.linalg.inv(Z.T@Z); h=np.einsum('ij,jk,ik->i',Z,bread,Z); residual=yy-Z@beta
    covariance=bread@(Z.T@((residual/(1-h))[:,None]**2*Z))@bread
    contrast=np.zeros(Z.shape[1]); contrast[1]=1; contrast[4]=data.G.mean() if group_mean is None else group_mean
    value=float(contrast@beta); se=float(np.sqrt(contrast@covariance@contrast))
    return {'estimate':value,'se':se,'lower':value-1.96*se,'upper':value+1.96*se,'n':len(data)}
models={'Unadjusted':None,'Observed_XG':(),'Oracle_XGU':('U',),'Observed_plus_mediator':('M',),'Oracle_plus_mediator':('U','M'),'Observed_plus_collider':('C',)}
rows=[]
for name,extra in models.items():
    if extra is None:
        a=observed.loc[observed.A==1,'Y']; b=observed.loc[observed.A==0,'Y']; value=float(a.mean()-b.mean()); se=float(np.sqrt(a.var()/len(a)+b.var()/len(b))); r={'estimate':value,'se':se,'lower':value-1.96*se,'upper':value+1.96*se,'n':len(observed)}
    else: r=estimate(oracle if 'U' in extra else observed,extra)
    rows.append({'method':name,**r,'difference_from_total_truth':r['estimate']-truth,'interpretation':'oracle direct effect' if name=='Oracle_plus_mediator' else ('identified total effect in simulation' if name=='Oracle_XGU' else 'association / unsuitable total-effect adjustment')})
estimates=pd.DataFrame(rows); table('estimates',estimates)
print('Oracle direct-effect target:',2+observed.G.mean())
fig,ax=plt.subplots(figsize=(9,4)); ax.errorbar(estimates.estimate,np.arange(len(estimates)),xerr=1.96*estimates.se,fmt='o'); ax.set_yticks(np.arange(len(estimates)),estimates.method); ax.axvline(truth,color='black',ls='--',label='Known sample total ATE'); ax.set_xlabel('Score contrast with 95% HC3 interval'); ax.legend(); figure('effect_comparison')
# Independent algebraic cross-check of the unadjusted contrast.
Z=np.column_stack([np.ones(len(observed)),observed.A]); ols=np.linalg.lstsq(Z,observed.Y,rcond=None)[0][1]
assert np.isclose(ols,estimates.loc[estimates.method=='Unadjusted','estimate'].iloc[0])


,method,estimate,se,lower,upper,n,difference_from_total_truth,interpretation
0,Unadjusted,5.073785,0.126202,4.826429,5.321140,2500,2.007385,association / unsuitable total-effect adjustment
1,Observed_XG,4.178831,0.106372,3.970342,4.387321,2500,1.112431,association / unsuitable total-effect adjustment
2,Oracle_XGU,3.007748,0.095041,2.821468,3.194027,2500,-0.058652,identified total effect in simulation
3,Observed_plus_mediator,2.690574,0.101137,2.492346,2.888802,2500,-0.375826,association / unsuitable total-effect adjustment
4,Oracle_plus_mediator,2.416867,0.093259,2.234078,2.599655,2500,-0.649533,oracle direct effect
5,Observed_plus_collider,3.185257,0.112916,2.963941,3.406573,2500,0.118857,association / unsuitable total-effect adjustment


Oracle direct-effect target: 2.5064


## 05 · Confounding sensitivity and negative control

Vary γ, the hidden variable’s influence on treatment selection, while keeping the outcome equations and total causal effect fixed. Two hundred independent samples per setting compare observed and oracle adjustment against each sample’s known standardized effect. Save every replicate, including poor coverage. Wilson intervals quantify simulation uncertainty even when observed coverage is zero; a plug-in Monte Carlo standard error of zero must not be read as certainty. Gamma=0 removes the treatment-side hidden-confounding path; this is a designed refutation condition rather than an empirical estimate of real confounding. A treatment association with Ypre is a negative-control warning because A cannot cause a preceding outcome. A null negative-control result would not establish exchangeability.

In [5]:

replicates=[]
for j,gamma in enumerate(protocol['gamma_grid']):
    for r in range(protocol['simulation_repetitions']):
        seed=np.random.SeedSequence([SEED,900,j,r]); data=generate(protocol['simulation_n'],gamma,seed); actual=float((data.Y1-data.Y0).mean())
        for name,extra in [('Observed_XG',()),('Oracle_XGU',('U',))]:
            fit=estimate(data,extra); replicates.append({'gamma':gamma,'replicate':r,'method':name,'truth':actual,**fit,'bias':fit['estimate']-actual,'covered':fit['lower']<=actual<=fit['upper']})
reps=pd.DataFrame(replicates); reps.to_csv(RUN/'sensitivity_replicates.csv',index=False)
rows=[]
for (gamma,name),g in reps.groupby(['gamma','method']):
    coverage=g.covered.mean(); ci=stats.binomtest(int(g.covered.sum()),len(g)).proportion_ci(method='wilson'); rows.append({'gamma':gamma,'method':name,'repetitions':len(g),'mean_bias':g.bias.mean(),'bias_MCSE':g.bias.std(ddof=1)/np.sqrt(len(g)),'RMSE':np.sqrt(np.mean(g.bias**2)),'coverage':coverage,'coverage_MCSE':np.sqrt(coverage*(1-coverage)/len(g)),'coverage_Wilson_lower':ci.low,'coverage_Wilson_upper':ci.high})
sensitivity=pd.DataFrame(rows); table('sensitivity_summary',sensitivity)
negative=pd.DataFrame([{'method':'Observed_XG',**estimate(observed,outcome='Ypre')},{'method':'Oracle_XGU',**estimate(oracle,('U',),outcome='Ypre')}]); table('negative_control',negative)
fig,ax=plt.subplots(1,2,figsize=(10,3.5))
for name,g in sensitivity.groupby('method'):
    ax[0].errorbar(g.gamma,g.mean_bias,yerr=1.96*g.bias_MCSE,fmt='o-',label=name); ax[1].plot(g.gamma,g.coverage,'o-',label=name)
ax[0].axhline(0,color='grey',ls='--'); ax[0].set(xlabel='Hidden-confounding strength gamma',ylabel='Mean bias ± 1.96 MCSE')
ax[1].axhline(.95,color='grey',ls='--'); ax[1].set(xlabel='Hidden-confounding strength gamma',ylabel='95% interval coverage',ylim=(-.03,1.03)); ax[0].legend(); figure('confounding_sensitivity')


,gamma,method,repetitions,mean_bias,bias_MCSE,RMSE,coverage,coverage_MCSE,coverage_Wilson_lower,coverage_Wilson_upper
0,0.0,Observed_XG,200,0.002012,0.010899,0.153768,0.950,0.015411,0.910422,0.972617
1,0.0,Oracle_XGU,200,-0.001149,0.009058,0.127779,0.970,0.012062,0.936106,0.986180
2,0.5,Observed_XG,200,0.705233,0.011473,0.723565,0.010,0.007036,0.002747,0.035722
3,0.5,Oracle_XGU,200,-0.005624,0.009832,0.138817,0.925,0.018625,0.879956,0.954025
4,1.0,Observed_XG,200,1.229948,0.010084,1.238147,0.000,0.000000,0.000000,0.018845
5,1.0,Oracle_XGU,200,0.000892,0.008707,0.122836,0.980,0.009899,0.949713,0.992196
6,1.5,Observed_XG,200,1.567438,0.010439,1.574340,0.000,0.000000,0.000000,0.018845
7,1.5,Oracle_XGU,200,-0.000253,0.010200,0.143892,0.960,0.013856,0.923068,0.979594


,method,estimate,se,lower,upper,n
0,Observed_XG,0.789780,0.057364,0.677346,0.902214,2500
1,Oracle_XGU,-0.019941,0.045154,-0.108443,0.068560,2500


## 06 · Heterogeneity and responsible decision memo

Report group-specific effects and treatment rates without treating a synthetic binary group as evidence about real demographic fairness. The total true effects are 2.56 and 3.56. A hypothetical intervention cost of 2.8 score-equivalent units makes net benefits differ, but the value conversion is invented for this simulation and cannot justify deployment. With observed-only confounding unresolved, recommend no automated allocation based on these estimates. A prospective, ethically reviewed evaluation would need a well-defined intervention, equitable access, consent and a defensible identification strategy. Provide human review and an appeal channel; stop or revise if harm, unequal access, poor overlap or assumption checks undermine the decision.

In [6]:

hetero=[]
for group in [0,1]:
    for name,data,extra in [('Observed_XG',observed,()),('Oracle_XGU',oracle,('U',))]:
        fit=estimate(data,extra,group_mean=group)
        hetero.append({'group':group,'method':name,**fit,'true_total_effect':2.56+group,'true_net_benefit_at_cost_2_8':2.56+group-2.8,'group_n':int((observed.G==group).sum()),'treatment_rate':float(observed.loc[observed.G==group,'A'].mean())})
table('heterogeneity',pd.DataFrame(hetero))
memo={'recommendation':'Do not deploy an allocation rule from the observed adjusted associations.','identification':'Hidden U leaves total effect unidentified from observed X,G. Oracle access is simulation-only.','fairness':'Inspect access and outcomes by relevant groups; synthetic G provides no evidence of real demographic performance. Do not turn the simulated group contrast into an exclusion rule.','cost':'2.8 synthetic score-equivalent units is a teaching assumption, not an observed economic utility.','appeal':'Any prospective pilot needs a documented human reconsideration route, explanation of decisions and correction of data errors.','stop_conditions':['evidence of harm or unequal access','poor overlap or unstable subgroup estimates','negative-control evidence inconsistent with identification assumptions','intervention differs from the stated treatment'],'next_evidence':'An ethically reviewed randomized or otherwise credibly identified prospective study, with explicit consent and monitoring.'}
save_json('responsible_decision_memo.json',memo)
obs=estimates[estimates.method=='Observed_XG'].iloc[0]; ora=estimates[estimates.method=='Oracle_XGU'].iloc[0]; cov=sensitivity[(sensitivity.gamma==1)&(sensitivity.method=='Observed_XG')].iloc[0]; nc=negative.iloc[0]
facts={'synthetic':True,'n':len(observed),'population_ATE':population_truth,'sample_ATE':truth,'observed_estimate':float(obs.estimate),'oracle_estimate':float(ora.estimate),'observed_coverage_gamma_1':float(cov.coverage),'negative_control_observed':float(nc.estimate),'identification':'Observed association only'}
summary=f"""This study uses an explicitly synthetic structural causal model to examine a training-participation claim. The intervention sets participation to one versus zero, and the estimand is the total population average effect on a synthetic score. The causal graph includes observed baseline variables, hidden ability, a mediator, and a post-treatment collider. The known population effect is {population_truth:.2f} points; the {len(observed)}-record sample has standardized truth {truth:.3f}.

Adjustment for observed baseline variables estimates {obs.estimate:.3f}, while an oracle regression that also observes hidden ability estimates {ora.estimate:.3f}. The first result is an adjusted association, because hidden confounding prevents identification from the observed variables. Mediator and collider adjustments illustrate different failures of an indiscriminate control strategy. A pre-treatment negative-control outcome has an observed adjusted participation contrast of {nc.estimate:.3f}, despite having no causal participation effect.

Two hundred independent replications at each of four confounding strengths retain bias, robust-interval coverage, and Monte Carlo uncertainty. At confounding strength one, observed-adjustment coverage is {cov.coverage:.1%}; the oracle provides a simulation benchmark rather than a deployable remedy. Group-specific effects and hypothetical costs demonstrate that an aggregate benefit does not settle allocation or fairness. The decision memo rejects automated allocation from unidentified associations and specifies equitable evaluation, human appeal, and stop conditions. These results concern the stated data-generating model only: there are no real participants, measured economic benefits, or empirical claims about actual training programs."""

zero=sensitivity[(sensitivity.gamma==0)&(sensitivity.method=='Observed_XG')].iloc[0]
ora_cov=sensitivity[(sensitivity.gamma==1)&(sensitivity.method=='Oracle_XGU')].iloc[0]
notes=f"""## Interpretation of this run

The observed adjustment overstates the known sample total effect by {obs.estimate-truth:.3f} score points. HC3 standard errors do not remove that identification error. At gamma zero, observed-adjustment coverage is {zero.coverage:.1%}; at gamma one it is {cov.coverage:.1%}, versus {ora_cov.coverage:.1%} for the oracle. Removing the hidden treatment-selection path restores the observed adjustment in the designed refutation setting.

Zero observed coverage in 200 replications is not an exact zero population probability: its Wilson upper confidence bound is {cov.coverage_Wilson_upper:.1%}. The negative-control association supplies a warning under this specified model, not a universal diagnostic test. The mediator-controlled oracle targets a direct effect, so comparing it as if it estimated the total effect would confuse two estimands. Group heterogeneity and cost assumptions do not solve the missing identification problem.
"""

(RUN/'DISCUSSION.md').write_text(notes)
display(Markdown(notes))



,group,method,estimate,se,lower,upper,n,true_total_effect,true_net_benefit_at_cost_2_8,group_n,treatment_rate
0,0,Observed_XG,3.407749,0.147605,3.118444,3.697055,2500,2.56,-0.24,1234,0.442464
1,0,Oracle_XGU,2.300389,0.127213,2.051053,2.549726,2500,2.56,-0.24,1234,0.442464
2,1,Observed_XG,4.930423,0.149016,4.638351,5.222495,2500,3.56,0.76,1266,0.516588
3,1,Oracle_XGU,3.697226,0.129706,3.443002,3.951451,2500,3.56,0.76,1266,0.516588


## Interpretation of this run

The observed adjustment overstates the known sample total effect by 1.112 score points. HC3 standard errors do not remove that identification error. At gamma zero, observed-adjustment coverage is 95.0%; at gamma one it is 0.0%, versus 98.0% for the oracle. Removing the hidden treatment-selection path restores the observed adjustment in the designed refutation setting.

Zero observed coverage in 200 replications is not an exact zero population probability: its Wilson upper confidence bound is 1.9%. The negative-control association supplies a warning under this specified model, not a universal diagnostic test. The mediator-controlled oracle targets a direct effect, so comparing it as if it estimated the total effect would confuse two estimands. Group heterogeneity and cost assumptions do not solve the missing identification problem.


## Completion and evidence

The following summary is computed from this run. The student verification and reflection are retained below. Automated execution is separate from student verification.

In [7]:
assert 150 <= len(summary.split()) <= 300, len(summary.split())
(RUN/'ABSTRACT.md').write_text('# Abstract\n\n'+summary+'\n')
save_json('summary.json', facts)
save_json('completed.json', {'completed_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'protocol_sha256':sha(PROTOCOL_PATH),'manifest_sha256':sha(BASE/'data/raw/manifest.json'),'summary_words':len(summary.split()),'analysis_source_sha256':sha(BASE/'analysis.py'),'notebook_code_sha256':notebook_code_hash(),'personal_verification':'Not asserted by automated execution'})
display(Markdown('## Results summary\n\n'+summary))
print('RESULT_DIR:', RUN.relative_to(BASE).as_posix())


## Results summary

This study uses an explicitly synthetic structural causal model to examine a training-participation claim. The intervention sets participation to one versus zero, and the estimand is the total population average effect on a synthetic score. The causal graph includes observed baseline variables, hidden ability, a mediator, and a post-treatment collider. The known population effect is 3.06 points; the 2500-record sample has standardized truth 3.066.

Adjustment for observed baseline variables estimates 4.179, while an oracle regression that also observes hidden ability estimates 3.008. The first result is an adjusted association, because hidden confounding prevents identification from the observed variables. Mediator and collider adjustments illustrate different failures of an indiscriminate control strategy. A pre-treatment negative-control outcome has an observed adjusted participation contrast of 0.790, despite having no causal participation effect.

Two hundred independent replications at each of four confounding strengths retain bias, robust-interval coverage, and Monte Carlo uncertainty. At confounding strength one, observed-adjustment coverage is 0.0%; the oracle provides a simulation benchmark rather than a deployable remedy. Group-specific effects and hypothetical costs demonstrate that an aggregate benefit does not settle allocation or fairness. The decision memo rejects automated allocation from unidentified associations and specifies equitable evaluation, human appeal, and stop conditions. These results concern the stated data-generating model only: there are no real participants, measured economic benefits, or empirical claims about actual training programs.

RESULT_DIR: results/20261125T111718437488Z


<div style="break-before: page"></div>

## Student independent verification and Reflection

I used a separate mathematical check  to verify the causal effect implied by the synthetic structural equations.

The check specifies that changing treatment A from zero to one changes the mediator by 0.7. Since the mediator contributes 0.8 to the outcome, the indirect effect is:

0.8 × 0.7 = 0.56

The direct effect is 2 + G, so the total individual effect is 2.56 + G.

With equal probabilities for G = 0 and G = 1, the population ATE is 3.06.

I also checked the sample-standardized effect using the saved group sizes: 1,234 records with G = 0 and 1,266 records with G = 1. The weighted result is 3.0664, matching the Notebook.

This confirms the structural calculation and its consistency with the saved group counts, although it does not independently rerun the simulated dataset.

The observed-variable adjustment estimated 4.179, which is noticeably above the known causal effect. I think this is the most important finding because it shows how hidden confounding can make an apparently reasonable regression result misleading.

I agree with using synthetic data here because the true effect is known. However, the findings demonstrate a methodological problem, not the effectiveness of a real training program.

My main takeaway is that obtaining a precise estimate is different from identifying the correct causal effect.
